# Real-Time Model Deployment in SageMaker AI

##  Learning Outcomes

After this video, you'll be able to:

<ul>
    <li>Configure deployment settings in SageMaker Studio</li>
    <li>Create a real-time endpoint for model serving</li>
    <li>Test endpoint performance with sample data</li>
    <li>Understand deployment best practices</li>
</ul>


## Preparing for Deployment

In [1]:
import sagemaker
from sagemaker import get_execution_role
import pandas as pd

# Initialize SageMaker session
session = sagemaker.Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Model Training with SageMaker

In [2]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role

bucket = session.default_bucket()
prefix = "ticketwise-model"
role = get_execution_role()

sklearn_estimator = SKLearn(
    entry_point="train_ticketwise_model.py",
    role=role,
    instance_type="ml.t3.large",
    framework_version="1.2-1",
    sagemaker_session=session,
    output_path=f"s3://{bucket}/{prefix}/output",
)

print(f"output paht is s3://{bucket}/{prefix}/output")
# Kick off training job
sklearn_estimator.fit()

INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2025-09-06-19-09-55-313


output paht is s3://sagemaker-us-east-1-350967407369/ticketwise-model/output
2025-09-06 19:09:56 Starting - Starting the training job...
2025-09-06 19:10:11 Starting - Preparing the instances for training...
2025-09-06 19:10:58 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-09-06 19:12:10,134 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-09-06 19:12:10,140 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-09-06 19:12:10,144 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-09-06 19:12:10,165 sagemaker_sklearn_contain

## Endpoint Creation

In [3]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

endpoint_name = "ticketwise-endpoint"

print(f"\nStarting model deployment to endpoint: {endpoint_name}...")

# The .deploy() method creates a real-time endpoint with the trained model.
# It provisions the infrastructure and makes the model available for requests.
predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
    endpoint_name=endpoint_name
)

print(f"Endpoint successfully deployed! Endpoint name: {predictor.endpoint_name}")

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2025-09-06-19-13-54-313



Starting model deployment to endpoint: ticketwise-endpoint...


INFO:sagemaker:Creating endpoint-config with name ticketwise-endpoint
INFO:sagemaker:Creating endpoint with name ticketwise-endpoint


------------!Endpoint successfully deployed! Endpoint name: ticketwise-endpoint


## Testing the Endpoint

In [4]:
# Load CSV directly from S3
df = pd.read_csv(f"s3://ticketwise-pipeline/ticketwise_dataset.csv").sample(frac=0.001, random_state=42)
print(df.shape)

# Select features and label
features = ["attachments", "previous_interactions", "contract_value", 
            "account_age_months", "open_tickets"]

print(f"\nSending sample data to the endpoint: \n {df[features]}")

# The .predict() method sends the data to the endpoint and returns the prediction.
prediction = predictor.predict(df[features])

print(f"\nPrediction received from the endpoint: {prediction}")

(5, 21)

Sending sample data to the endpoint: 
       attachments  previous_interactions  contract_value  account_age_months  \
1501            1                      3         5417.06                  27   
2586            0                      3          115.70                   7   
2653            2                      1         5912.09                  36   
1055            0                      3         1261.97                   9   
705             1                      4         2728.12                  35   

      open_tickets  
1501             1  
2586             1  
2653             3  
1055             0  
705              0  

Prediction received from the endpoint: [['0'], ['0'], ['0'], ['1'], ['0']]


In [5]:
print("Deleting the endpoint to avoid charges")

try:
    predictor.delete_endpoint()
    print("Endpoint successfully deleted.")
except Exception as e:
    print(f"Error deleting the endpoint. Please delete it manually in the SageMaker console. Error: {e}")

INFO:sagemaker:Deleting endpoint configuration with name: ticketwise-endpoint


Deleting the endpoint to avoid charges


INFO:sagemaker:Deleting endpoint with name: ticketwise-endpoint


Endpoint successfully deleted.


## Recap


In this video, we covered:

- SageMaker deployment configuration

- Real-time endpoint creation

- Endpoint testing and validation

- Best practices for deployment
